# Reproducing Mah et al. (2023) from dandiset 001169

**Paper.** Mah A, Schiereck SS, Bossio V, Constantinople CM (2023). Distinct value computations
support rapid sequential decisions. *Nature Communications* 14:7573.
[doi:10.1038/s41467-023-43250-x](https://doi.org/10.1038/s41467-023-43250-x). This attempt used
the full text, which is open access (PMC10663503), and the paper's analysis code
(github.com/constantinoplelab/published, `rat_behavior`, commit b7b68ca).

**Dandiset.** [DANDI:001169](https://dandiarchive.org/dandiset/001169/0.260825.1909), version
0.260825.1909. The paper cites no DANDI version (its data statement points to Zenodo); this is
the only published version, and the draft has the same 23,034 assets and size, so it is pinned.

**What the dandiset holds.** One NWB file per session: 23,034 sessions of 291 rats in the
temporal wagering task, with the raw Bpod states, events and actions, the raw trials table, and
a table of processed trials that the conversion filled from the paper's per-rat structs
(`A_Structs_Final` on Zenodo). No neural data; the paper has none.

**Outside inputs.** The paper's processed structs on Zenodo
([10.5281/zenodo.10031483](https://doi.org/10.5281/zenodo.10031483)) are read for one claim,
`zenodo-link`, which compares them with the dandiset. The Figure 4 claims use the structs of all
final-stage sessions (`A_Structs_all`), which the dandiset does not hold; they are in the second
notebook, `reproduce_mah_2023_fig4.ipynb`.

Each file is about 1 MB and is read whole into memory, in parallel; loading takes about nine
minutes. Random numbers (exponential and model fits from random starts, shuffles) come from a
generator per claim section with a fixed seed.

| Claim | Data | Status |
|---|---|---|
| dataset-counts | dandiset | tested |
| zenodo-link | with inputs (Zenodo) | tested |
| fig1f-wait-time-block | dandiset | tested |
| fig1i-initiation-block | dandiset | tested |
| figs2-unrewarded-drive | dandiset | tested |
| figs12-no-sex-difference | dandiset | tested |
| fig2a-wait-time-converges | dandiset | tested |
| fig2b-initiation-previous-block | dandiset | tested |
| fig2cd-regression-kernels | dandiset | tested |
| fig2e-tau-differ | dandiset | tested |
| fig2f-tau-uncorrelated | dandiset | tested |
| fig2gh-fast-slow-groups | dandiset | tested |
| fig2j-wait-time-previous-reward | dandiset | tested |
| fig2k-initiation-previous-reward | dandiset | tested |
| fig2l-sensitivity-differs | dandiset | tested |
| fig3c-catch-probability | dandiset | tested |
| fig3j-model-comparison | dandiset | tested |
| figs9-misinference | dandiset | tested |
| fig3l-inference-quality | dandiset | tested |
| fig3h-dynamic-learning-rate | dandiset | blocked: methods underspecified |
| fig4a-wait-ratio-learning | elsewhere (Zenodo) | tested in the second notebook |
| fig4bc-regression-learning | elsewhere (Zenodo) | tested in the second notebook |
| fig4de-overshoot-learning | dandiset | blocked: data missing |
| fig5b-precue-offer | dandiset | tested |
| fig5c-precue-previous-reward | dandiset | tested |
| fig5d-precue-previous-block | dandiset | tested |

In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[var] = "1"

import time
import warnings
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import wagering as w

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

started = time.time()
assets = w.asset_list()
sessions_all, trials_all = w.load_all(assets)
print(f"loaded {len(sessions_all)} files ({assets['size'].sum() / 1e9:.1f} GB) from dandiset "
      f"{w.DANDISET_ID} version {w.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
S, T = sessions_all, trials_all
print("Assets:", len(assets), "; all NWB:", bool(assets.path.str.endswith(".nwb").all()))
print("Subjects (folders):", assets.path.str.split("/").str[0].nunique(), "; subject ids in files:", S.subject_id.nunique())
print("Sessions:", len(S), "; trials:", len(T))
print("Species in files:", S.species.unique().tolist())
print("Sex per rat:", S.groupby("subject_id").sex.agg(lambda s: "/".join(sorted(set(s)))).value_counts().to_dict())
print("Protocol per session:", S.protocol.value_counts().to_dict())
print("Training stage values in processed trials:", T.training_stage.value_counts().to_dict())
print("Subjects with an age in the files:", S[S.has_age].subject_id.nunique(), "of", S.subject_id.nunique())
print("Processed and raw trial tables have the same length in every file:", bool((S.n_raw_trials == S.n_trials).all()))
print("Columns of processed_trials (same in every file):", S.processed_columns.nunique() == 1)
print("  ", list(S.processed_columns.iloc[0]))
print("Session start times:", S.start.min(), "to", S.start.max())

The dandiset's summary (291 subjects, *Rattus norvegicus*) agrees with the files, and every rat
has one sex. The processed trials give the training stage as 9 for every trial; the paper says
all data are from stage 8, and the Zenodo README says that code 9 is the paper's stage 8, which
the NWB files do not say. Ages are absent for 275 of 291 rats (the Methods give 6 to 24 months).
The processed table has no catch probability: the conversion's column mapping spells the
struct's `prob_catch` as `prob_cacth`, so it was not copied. The raw trials table has
`catch_percentage`, which is used in its place (compared with the struct in `zenodo-link`).

In [ ]:
# Recordings deposited twice: sessions of the same rat identical trial for trial.
sessions, trials, removed = w.drop_duplicate_sessions(sessions_all, trials_all)
print("Sessions identical to another of the same rat (removed):")
print(removed[["subject_id", "session_id", "n_trials", "path"]].to_string(index=False))
kept_twin = sessions[(sessions.subject_id.isin(removed.subject_id)) &
                     (sessions.start.isin(removed.start))]
print("Kept:")
print(kept_twin[["subject_id", "session_id", "n_trials", "path"]].to_string(index=False))
sessions["date"] = sessions.session_id.str.split("-").str[1]
same_day = sessions[sessions.duplicated(["subject_id", "date"], keep=False)]
print(f"\nRats with two different sessions on one date: {same_day.subject_id.nunique()} "
      f"({len(same_day)} sessions); their trial counts differ, so these are separate sessions.")
print("Session ids with a suffix:", sessions_all[sessions_all.session_id.str.count("-") > 2].session_id.tolist())

One session is deposited twice: rat L011's session of 2022-10-04 appears as
`RWTautowait2-20221004-095612` and `RWTautowait2-20221004-095612-x`, with identical trials. It
is counted once from here on. The other same-day pairs are different sessions. The session
`L004 ... -edit` has no twin.

In [ ]:
print("Offered volumes (uL) by sex, trials:")
print(pd.crosstab(T.reward_volume_ul, sessions_all.sex.to_numpy()[T.session]).to_string())
min_vol = T.groupby("rat", observed=True).reward_volume_ul.min()
sex = sessions_all.groupby("subject_id").sex.first()
print("Rats whose smallest offer is 4 uL, by sex:", sex[min_vol[min_vol == 4].index].value_counts().to_dict(),
      "; 5 uL:", sex[min_vol[min_vol == 5].index].value_counts().to_dict())
print("\nOffers (standardised) per block, trials:")
print(pd.crosstab(trials.block_type.map({1: "mixed", 2: "high", 3: "low"}), trials.reward).to_string())
# Block length in completed (non-violation) trials, for blocks that end inside a session.
b = trials.block_type.to_numpy(); s = trials.session.to_numpy(); v = trials.is_violation.to_numpy()
new = np.r_[True, (b[1:] != b[:-1]) | (s[1:] != s[:-1])]
block_id = np.cumsum(new)
ends_in_session = np.r_[(s[1:] == s[:-1]) & (b[1:] != b[:-1]), False]
completed = pd.Series(~v).groupby(block_id).sum()
closed = pd.Series(ends_in_session).groupby(block_id).any()
print("\nCompleted trials in blocks that end within a session:", completed[closed].describe()[["count", "min", "50%", "max"]].to_dict())
print("Share of such blocks with exactly 40 completed trials:", round(float((completed[closed] == 40).mean()), 3))
first_block = trials.groupby("session").block_type.first()
print("First block of each session:", first_block.map({1: "mixed", 2: "high", 3: "low"}).value_counts().to_dict())
pc = trials.catch_percentage.astype(float).round(3)
print("\nCatch probability (raw trials, catch_percentage):", pc.value_counts().sort_index().to_dict())
print("Share of catch trials at each catch probability:", trials.is_catch.groupby(pc).mean().round(3).to_dict())

Offers are 5 to 80 uL for 192 rats and 4 to 64 uL for 99 female rats, as the Methods describe
for some females; they are standardised to 5 to 80 uL below. Low blocks offer only 5, 10 and
20 uL, high blocks 20, 40 and 80 uL, mixed blocks all five. Blocks change after a median of 40
completed trials, as the Methods state, but 456 of 23,034 sessions start with a high or low block
where the Methods say the first block is always mixed. The files'
catch probabilities include 0.30, which the paper's list of cohorts (0.1, 0.15, 0.2, 0.25 and
0.35) does not have; the realised share of catch trials follows each stated probability.

In [ ]:
wt = trials.wait_time.to_numpy(); wtu = trials.wait_time_unthresholded.to_numpy(); thr = trials.wait_time_threshold.to_numpy()
both = ~np.isnan(wtu)
print("Trials with a wait time (unthresholded):", int(both.sum()), "; opt-out or rewarded among them:",
      round(float((trials.is_opt_out | trials.is_rewarded)[both].mean()), 4))
print("Wait time equals the unthresholded value where that is at most the threshold:",
      bool(np.allclose(wt[both & (wtu <= thr)], wtu[both & (wtu <= thr)])),
      "; NaN where above:", bool(np.isnan(wt[both & (wtu > thr)]).all()), f"({int((both & (wtu > thr)).sum())} trials)")
print("Thresholds per rat (s):", trials.groupby("rat", observed=True).wait_time_threshold.first().describe()[["min", "50%", "max"]].round(1).to_dict())
print("One threshold per rat:", bool((trials.groupby("rat", observed=True).wait_time_threshold.nunique() == 1).all()))
print("Unthresholded wait times (s):", np.nanpercentile(wtu, [0, 50, 99, 100]).round(2).tolist())
iti = trials.inter_trial_interval.to_numpy()
print("Trial initiation times (s), percentiles 0, 50, 99, 100:", np.nanpercentile(iti, [0, 50, 99, 100]).round(2).tolist(),
      "; missing:", int(np.isnan(iti).sum()))

`wait_time` is the unthresholded wait time with values above the rat's threshold removed, one
threshold per rat, as the descriptions say (the Methods say one standard deviation above the
pooled mean). Wait times and trial initiation times are in seconds; the largest initiation times
(minutes to hours) are pauses, which the analysis removes at each rat's 99th percentile, as the
Methods do.

In [ ]:
# The processed times against the raw Bpod events of the same files, for a sample of sessions.
import h5py, io, requests

def raw_check(item):
    session_id, url = item
    content = requests.get(url, timeout=120).content
    with h5py.File(io.BytesIO(content), "r") as f:
        rec = f["acquisition/task_recording"]
        st_names = np.array([x.decode() for x in f["general/task/state_types/state_name"][:]])
        ev_names = np.array([x.decode() for x in f["general/task/event_types/event_name"][:]])
        states = pd.DataFrame({"name": st_names[rec["states/state_type"][:]], "start": rec["states/start_time"][:]})
        ev = pd.DataFrame({"name": ev_names[rec["events/event_type"][:]], "t": rec["events/timestamp"][:],
                           "value": [x.decode() for x in rec["events/value"][:]]})
        tstart, tstop = f["intervals/trials/start_time"][:], f["intervals/trials/stop_time"][:]
        g = f["intervals/processed_trials"]
        proc = {k: g[k][:] for k in ("inter_trial_interval", "wait_time_unthresholded", "is_rewarded", "is_catch", "start_time")}
    cin = np.sort(ev[(ev.name == "center_port_poke") & (ev.value == "In")].t.to_numpy())
    first_in = np.array([cin[(cin >= a) & (cin < b)].min() if ((cin >= a) & (cin < b)).any() else np.nan
                         for a, b in zip(tstart, tstop)])
    prev_unrewarded = np.r_[False, ~proc["is_rewarded"][:-1]]
    d_iti = (first_in - tstart - proc["inter_trial_interval"])[prev_unrewarded]
    trial_of = lambda t: np.searchsorted(tstart, t, side="right") - 1
    side = states[states.name == "wait_for_side_poke"]; opt = states[states.name == "opt_out"]
    side = pd.Series(side.start.to_numpy(), index=trial_of(side.start.to_numpy())).groupby(level=0).first()
    opt = pd.Series(opt.start.to_numpy(), index=trial_of(opt.start.to_numpy())).groupby(level=0).first()
    waited = (opt - side).dropna()
    waited = waited[proc["is_catch"][waited.index.to_numpy()]]   # catch trials: the rat waited until it opted out
    d_wt = waited.to_numpy() - proc["wait_time_unthresholded"][waited.index.to_numpy()]
    return {"session": session_id, "same_trial_starts": bool(np.allclose(tstart, proc["start_time"])),
            "n_iti": int(np.sum(~np.isnan(d_iti))), "max_abs_iti_diff_s": np.nanmax(np.abs(d_iti)),
            "n_catch_optout": len(d_wt), "max_abs_wait_diff_s": np.nanmax(np.abs(d_wt)) if len(d_wt) else np.nan}

sample = sessions_all.merge(assets[["path", "url"]], on="path").sample(24, random_state=0)
with ProcessPoolExecutor(4) as pool:
    checks = pd.DataFrame(list(pool.map(raw_check, zip(sample.session_id, sample.url))))
print(checks.to_string(index=False))

In 24 sessions drawn at random, the processed trial initiation time after an unrewarded trial
equals the time from the trial's start to its first center poke in the raw events, and the
processed wait time of a catch trial equals the time from the side light (`wait_for_side_poke`)
to the opt-out (`opt_out`), to within the precision printed; the processed and raw trials start
at the same times. The processed measures and the raw events therefore share one clock. The
block contrast itself (initiation and wait times by block, `fig1f` and `fig1i` below) shows
that the trial labels line up with the behaviour.

In [ ]:
cue = sessions[sessions.center_port_cue == "True"]
print("Sessions with center_port_cue true:", len(cue), "in", cue.subject_id.nunique(), "rats; with it false:",
      int((sessions.center_port_cue == "False").sum()), "; argument absent:", int((sessions.center_port_cue == "absent").sum()))
FIG5_RATS = ["M041", "M042", "M043", "M044", "M045", "M046", "M047", "M048", "G070", "E005", "E008",
             "G078", "G076", "G069", "S042", "S050"]   # the rats listed in the paper's Figure5.m
print("Same rats as Figure5.m:", sorted(cue.subject_id.unique()) == sorted(FIG5_RATS))
per_rat = sessions[sessions.subject_id.isin(FIG5_RATS)].groupby("subject_id").agg(
    cue_sessions=("center_port_cue", lambda s: int((s == "True").sum())),
    other_sessions=("center_port_cue", lambda s: int((s != "True").sum())),
    first_cue=("start", lambda s: None))
for r in per_rat.index:
    ss = sessions[sessions.subject_id == r]
    per_rat.loc[r, "first_cue"] = ss[ss.center_port_cue == "True"].start.min().date()
    per_rat.loc[r, "last_other"] = ss[ss.center_port_cue != "True"].start.max().date() if (ss.center_port_cue != "True").any() else None
print(per_rat.to_string())
print("\nTrials in cue sessions:", int(trials.session.isin(cue.index).sum()), "of", len(trials))

The 762 sessions flagged with the task argument `center_port_cue` belong to exactly the 16 rats
that the paper's Figure 5 code lists, and in each rat they follow its sessions of the original
task, so the flag is consistent with the pre-initiation cue task; four rats (M045 to M048) have
only cue sessions, as the Methods say four rats were trained only on it. The raw files cannot
confirm the flag: every sound is stored as the same action ("sound_output", "On") without its
identity. These sessions are part of the paper's main data (`A_Structs_Final`, from which the
conversion was made), so they enter the analyses of Figures 1 to 3 as in the paper; the Methods
do not say so.

In [ ]:
rats = w.rat_arrays(trials, sessions)
names = sorted(rats)
sex_of = sessions.groupby("subject_id").sex.first()
print(len(names), "rats;", "trials per rat:", pd.Series({r: len(rats[r]["reward"]) for r in names}).describe()[["min", "50%", "max"]].to_dict())

## dataset-counts

> Data from 291 Long-Evans rats (184 male, 107 female), with a median of 71 sessions and 33,493
> trials per rat.

In [ ]:
per = sessions.groupby("subject_id").agg(sessions=("n_trials", "size"), trials=("n_trials", "sum"))
counts = {"rats": len(per), "median sessions per rat": per.sessions.median(), "median trials per rat": per.trials.median(),
          "male rats": int((sex_of == "M").sum()), "female rats": int((sex_of == "F").sum())}
expected = {"rats": 291, "median sessions per rat": 71, "median trials per rat": 33493, "male rats": 184, "female rats": 107}
print(pd.DataFrame({"dandiset": counts, "paper": expected}).to_string())
per_all = sessions_all.groupby("subject_id").agg(sessions=("n_trials", "size"), trials=("n_trials", "sum"))
print("\nWith the duplicated L011 session counted twice: median sessions", per_all.sessions.median(),
      ", median trials", per_all.trials.median())
print("Sessions per rat, range:", per.sessions.min(), "to", per.sessions.max(), "; trials per rat:", per.trials.min(), "to", per.trials.max())
print("The Methods also give 160 male and 114 female rats (274 in all).")

**Result: reproduced.** The files hold 291 rats, 184 male and 107 female, with a median of 71
sessions and 33,493 trials per rat, each session counted once; all five numbers match the paper.
The other count in the Methods (160 male and 114 female, 274 rats) matches neither the files nor
the paper's 291.

## zenodo-link

> The dandiset holds the paper's processed per-rat data (A_Structs_Final in the Zenodo deposit)
> for every rat, session and trial, with the same values.

The structs are read from the Zenodo zip archive with range requests (about 0.4 GB).

In [ ]:
started = time.time()
structs = w.load_zenodo_structs("A_Structs_Final")
print(f"read {len(structs)} structs in {time.time() - started:.0f} s")
FIELDS = {"reward": "volume", "block": "block", "catch": "catch", "vios": "vios", "hits": "hits",
          "optout": "optout", "wait_time": "wait_time", "ITI": "iti", "prob_catch": "prob_catch"}

EXACT = ["reward", "block", "catch", "vios", "hits", "optout"]
CLOSE = {"wait_time": 1e-4, "ITI": 1e-4, "prob_catch": 1e-6}   # float32 in the NWB files

def sessions_of(get, bounds):
    # Per session: a key from its trial count and integer fields, and its float fields.
    out = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        key = str(b - a) + "|" + "|".join(np.asarray(get(f)[a:b], float).astype(np.int64).tobytes().hex() for f in EXACT)
        out.append((key, {f: np.asarray(get(f)[a:b], float) for f in CLOSE}))
    return out

rows = []
for r in sorted(set(structs) | set(names)):
    Z, A = structs.get(r), rats.get(r)
    row = {"rat": r, "in_zenodo": Z is not None, "in_dandiset": A is not None}
    if Z is not None and A is not None:
        nt_z = np.atleast_1d(Z["ntrials"]).astype(int)
        nt_d = np.bincount(A["session"])
        sz = sessions_of(lambda f: np.asarray(Z[f], float).ravel(), np.r_[0, np.cumsum(nt_z)])
        sd = sessions_of(lambda f: A[FIELDS[f]].astype(float), np.r_[0, np.cumsum(nt_d)])
        pool_d = {}
        for k, v in sd:
            pool_d.setdefault(k, []).append(v)
        matched, unmatched_z, float_bad = 0, [], 0
        for k, v in sz:
            cands = pool_d.get(k, [])
            hit = next((i for i, c in enumerate(cands) if all(np.allclose(c[f], v[f], atol=t, equal_nan=True) for f, t in CLOSE.items())), None)
            if hit is None:
                unmatched_z.append(int(k.split("|")[0]))
                float_bad += bool(cands)
            else:
                cands.pop(hit); matched += 1
        row.update(sessions_zenodo=len(nt_z), sessions_dandiset=len(nt_d),
                   same_order=bool(len(sz) == len(sd) and all(a[0] == b[0] for a, b in zip(sz, sd)) and len(unmatched_z) == 0),
                   only_zenodo=len(unmatched_z), only_dandiset=sum(len(v) for v in pool_d.values()),
                   trials_only_zenodo=int(sum(unmatched_z)), float_mismatch=float_bad)
        row["same_set"] = row["only_zenodo"] == 0 and row["only_dandiset"] == 0
    rows.append(row)
link = pd.DataFrame(rows)
print("Rats in both:", int((link.in_zenodo & link.in_dandiset).sum()), "; only Zenodo:", link[~link.in_dandiset].rat.tolist(),
      "; only dandiset:", link[~link.in_zenodo].rat.tolist())
print("Rats whose sessions match in order, trial counts and every compared value (criterion):", int(link.same_order.sum()))
print("Rats whose sessions match as a set, ignoring order:", int(link.same_set.sum()))
print("\nRats that differ:")
print(link[~link.same_order][["rat", "sessions_zenodo", "sessions_dandiset", "same_set", "only_zenodo", "only_dandiset",
                               "trials_only_zenodo", "float_mismatch"]].to_string(index=False))
print("\nSessions on Zenodo with no identical session in the dandiset:", int(link.only_zenodo.sum()),
      f"({int(link.trials_only_zenodo.sum())} trials, {link.trials_only_zenodo.sum() / sum(len(np.asarray(z['reward']).ravel()) for z in structs.values()):.5f} of all);",
      "sessions in the dandiset with no identical one on Zenodo:", int(link.only_dandiset.sum()))

In [ ]:
nz = len(np.atleast_1d(structs["L011"]["ntrials"]))
print("L011 sessions on Zenodo:", nz, "; in the dandiset with the duplicate:", int((sessions_all.subject_id == "L011").sum()))
# Where the order differs: the sessions of the dandiset in start-time order, against the structs' order.
for r in link[(~link.same_order) & link.same_set].rat:
    nt_z = np.atleast_1d(structs[r]["ntrials"]).astype(int); nt_d = np.bincount(rats[r]["session"])
    i = int(np.flatnonzero(nt_z != nt_d)[0])
    ss = sessions[sessions.subject_id == r].reset_index(drop=True)
    moved = ss[ss.n_trials == nt_d[i]].session_id.tolist()
    print(f"{r}: first different position {i}; the dandiset's session there ({moved[0]}, {nt_d[i]} trials) is at position "
          f"{int(np.flatnonzero(nt_z == nt_d[i])[0])} in the struct")

**Result: partially reproduced.** All 291 rats are in both. 273 rats (94%) match the Zenodo
structs in full, session by session in the same order with the same trial counts and values, at
least the 90% the criterion asks for partial reproduction, not all 291. Ignoring order, 283 rats
match. Nine sessions of the structs (3,645 trials, 0.035% of all) are missing from the dandiset,
in eight rats (G001 two, G022, G026, G052, G060, J011, J023, S028). In 10 rats the sessions are
the same but in another order: the dandiset orders them by start time, and the structs place
some sessions later (for several rats, those of 2020-06-22). Every session in the dandiset has an
identical session on Zenodo (after the duplicate of L011 is removed; the L011 struct has 204
sessions, the dandiset 205), and in matched sessions the raw trials' catch probability equals the
structs' `prob_catch`.

## fig1f-wait-time-block

> Rats waited about 10% less for 20 uL in high blocks than in low blocks (p = 8.26e-49,
> two-tailed Wilcoxon signed-rank test, N = 291).

In [ ]:
wc = {r: w.wait_curves(rats[r]) for r in names}
hi = np.array([wc[r][(2, 20)] for r in names]); lo = np.array([wc[r][(3, 20)] for r in names])
zhi = np.array([wc[r][("z", 2, 20)] for r in names]); zlo = np.array([wc[r][("z", 3, 20)] for r in names])
ratio = hi / lo
print(f"(a) 1 - mean of per-rat ratios: {100 * (1 - np.nanmean(ratio)):.2f}%   (b) 1 - ratio of population means: "
      f"{100 * (1 - np.nanmean(hi) / np.nanmean(lo)):.2f}%   [expected 10 +- 2.5]")
print(f"Signed-rank, per-rat means of z-scored wait times (Methods): p = {w.signrank(zhi, zlo):.3g}; "
      f"high below low in {np.sum(zhi < zlo)} of {np.sum(~np.isnan(zhi - zlo))} rats")
print(f"Variant, raw means (Figure1.m): p = {w.signrank(hi, lo):.3g}; high below low in {np.sum(hi < lo)} rats")
pw = np.array([wc[r]["p"] for r in names])
print(f"Rats with p < 0.05 (one-tailed rank-sum, as the code): {np.sum(pw < 0.05)} of {len(names)}")
print("Per-rat ratio, range:", np.nanmin(ratio).round(3), "to", np.nanmax(ratio).round(3), "; median", np.nanmedian(ratio).round(3))
for s_ in ("M", "F"):
    m = np.array([sex_of[r] == s_ for r in names])
    print(f"  {s_}: {m.sum()} rats, mean ratio {np.nanmean(ratio[m]):.3f}, signed-rank p = {w.signrank(zhi[m], zlo[m]):.3g}")
no_cue = {r: w.subset(rats[r], ~rats[r]["cue"]) for r in names}
wc_nc = {r: w.wait_curves(no_cue[r]) for r in names if len(no_cue[r]["reward"])}
r_nc = np.array([wc_nc[r][(2, 20)] / wc_nc[r][(3, 20)] for r in wc_nc])
print(f"Variant without the pre-cue sessions: mean ratio {np.nanmean(r_nc):.3f} over {np.sum(~np.isnan(r_nc))} rats")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for b, c, lab in ((1, "k", "mixed"), (2, "r", "high"), (3, "b", "low")):
    m = np.array([[wc[r][(b, v)] for v in w.STANDARD_VOLUMES] for r in names])
    axes[0].errorbar(range(5), np.nanmean(m, 0), np.nanstd(m, 0) / np.sqrt(np.sum(~np.isnan(m), 0)), color=c, label=lab)
axes[0].set_xticks(range(5), w.STANDARD_VOLUMES); axes[0].set_xlabel("offer (uL)"); axes[0].set_ylabel("mean wait time (s)"); axes[0].legend()
axes[1].hist(ratio[~np.isnan(ratio)], 30, color="0.6"); axes[1].axvline(1, color="k", ls="--"); axes[1].axvline(np.nanmean(ratio), color="k")
axes[1].set_xlabel("wait time ratio, 20 uL high/low"); axes[1].set_ylabel("rats")
plt.tight_layout(); plt.show()
print("All per-rat ratios are inside the histogram's axis.")

**Result: reproduced.** Rats waited 10.58% (reading a) or 10.79% (reading b) less for 20 uL
in high than in low blocks, within 10 +- 2.5, and the signed-rank test on per-rat means of
z-scored wait times finds high below low (p = 4.6e-47, 279 of 291 rats). The code's test on raw
means gives p = 8.2e-47, against the paper's 8.26e-49. The effect holds in males (p = 3.8e-30)
and females (p = 1.2e-18).

## fig1i-initiation-block

> Rats initiated trials more quickly in high than in low blocks (p = 1.82e-49, two-tailed
> Wilcoxon signed-rank test, N = 291).

In [ ]:
ib = {r: w.initiation_by_block(rats[r]) for r in names}
ih = np.array([ib[r][2] for r in names]); il = np.array([ib[r][3] for r in names])
zih = np.array([ib[r][("z", 2)] for r in names]); zil = np.array([ib[r][("z", 3)] for r in names])
print(f"Signed-rank, per-rat means of z-scored initiation times (Methods): p = {w.signrank(zih, zil):.3g}; "
      f"high below low in {np.sum(zih < zil)} of {len(names)} rats")
print(f"Variant, raw means (Figure1.m): p = {w.signrank(ih, il):.3g}")
print(f"Mean ratio high/low {np.mean(ih / il):.3f} (range {np.min(ih / il):.3f} to {np.max(ih / il):.3f})")
pi_ = np.array([ib[r]["p"] for r in names])
print(f"Rats with p < 0.05 (one-tailed rank-sum, as the code): {np.sum(pi_ < 0.05)} of {len(names)}")
print("Smallest p the two-sided signed-rank test can give for 291 rats all in one direction:",
      f"{w.signrank(np.ones(291) + np.arange(291) * 1e-6):.3g}")
for s_ in ("M", "F"):
    m = np.array([sex_of[r] == s_ for r in names])
    print(f"  {s_}: mean ratio {np.mean(ih[m] / il[m]):.3f}, signed-rank p = {w.signrank(zih[m], zil[m]):.3g}")

**Result: reproduced.** 290 of 291 rats initiated trials faster in high than in low blocks;
the signed-rank test on z-scored means gives p = 1.82e-49, the paper's value, and on raw means
p = 1.84e-49. The mean ratio high/low is 0.562.

## figs2-unrewarded-drive

> Block sensitivity of trial initiation times is largely driven by initiation times that follow
> unrewarded trials.

In [ ]:
po = {r: w.initiation_by_previous_outcome(rats[r]) for r in names}
d_rew = np.array([po[r]["rewarded"][0] - po[r]["rewarded"][1] for r in names])
d_unrew = np.array([po[r]["unrewarded"][0] - po[r]["unrewarded"][1] for r in names])
print(f"Low minus high, z-scored initiation time; after rewarded: median {np.nanmedian(d_rew):.3f}, "
      f"after unrewarded: median {np.nanmedian(d_unrew):.3f}")
print(f"Signed-rank, after unrewarded against after rewarded: p = {w.signrank(d_unrew, d_rew):.3g}; "
      f"larger after unrewarded in {np.sum(d_unrew > d_rew)} of {np.sum(~np.isnan(d_unrew - d_rew))} rats")
print(f"Each against zero: after rewarded p = {w.signrank(d_rew):.3g}, after unrewarded p = {w.signrank(d_unrew):.3g}")

**Result: reproduced.** The block difference in z-scored initiation time (low minus high) is
larger after unrewarded trials (median 0.347) than after rewarded ones (median -0.017) in 291 of
291 rats, p = 1.8e-49 by signed rank.

## figs12-no-sex-difference

> Male and female rats did not differ in their contextual effects (wait time and trial
> initiation time ratios).

In [ ]:
male = np.array([sex_of[r] == "M" for r in names])
lat_ratio = ih / il
print(f"Wait time ratio: males median {np.nanmedian(ratio[male]):.3f} ({male.sum()}), females median "
      f"{np.nanmedian(ratio[~male]):.3f} ({(~male).sum()}); rank-sum p = {w.ranksum(ratio[male], ratio[~male]):.3g}")
print(f"Initiation time ratio: males median {np.median(lat_ratio[male]):.3f}, females median "
      f"{np.median(lat_ratio[~male]):.3f}; rank-sum p = {w.ranksum(lat_ratio[male], lat_ratio[~male]):.3g}")

**Result: reproduced.** Males and females do not differ in the wait time ratio (medians 0.887
and 0.892, rank-sum p = 0.24) or in the initiation time ratio (medians 0.552 and 0.540, p = 0.56).

## fig2a-wait-time-converges

> At transitions from high or low blocks into mixed blocks, wait times converge to a common
> value regardless of the previous block.

## fig2b-initiation-previous-block

> Trial initiation times in mixed blocks depend strongly on the previous block, unlike wait times.

In [ ]:
wd = {r: w.wait_dynamics(rats[r]) for r in names}
idn = {r: w.initiation_dynamics(rats[r]) for r in names}
idn_raw = {r: w.initiation_dynamics(rats[r], detrend=False) for r in names}
x = np.arange(-20, 21)
PRE, POST = (x >= -20) & (x <= -6), (x >= 11) & (x <= 20)

def transition_summary(dyn, label):
    diff = np.array([dyn[r][-2] - dyn[r][-1] for r in dyn])     # low-to-mixed minus high-to-mixed
    m = np.nanmean(diff, 0)
    pre, post = np.nanmean(m[PRE]), np.nanmean(m[POST])
    per_rat_post = np.nanmean(diff[:, POST], 1)
    print(f"{label}: before {pre:.3f}, after {post:.3f}, after/before {post / pre:.3f}; per-rat after: median "
          f"{np.nanmedian(per_rat_post):.3f}, above zero in {np.sum(per_rat_post > 0)} of {np.sum(~np.isnan(per_rat_post))}, "
          f"signed-rank p = {w.signrank(per_rat_post):.3g}")
    late = (x >= 6) & (x <= 20)
    print(f"   variant window +6 to +20: after/before {np.nanmean(m[late]) / pre:.3f}")
    return diff, m

wd_diff, wd_m = transition_summary(wd, "Wait time (fig2a)")
id_diff, id_m = transition_summary(idn, "Initiation time (fig2b)")
_ = transition_summary(idn_raw, "Initiation time, not detrended (variant)")
for s_, m_ in (("M", male), ("F", ~male)):
    for lab, dd in (("wait", wd_diff), ("initiation", id_diff)):
        mm = np.nanmean(dd[m_], 0)
        print(f"  {s_} {lab}: after/before {np.nanmean(mm[POST]) / np.nanmean(mm[PRE]):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, dyn, lab in ((axes[0], wd, "delta z wait time"), (axes[1], idn, "z initiation time (detrended)")):
    for kind, c in ((-2, "b"), (-1, "r")):
        arr = np.array([dyn[r][kind] for r in names])
        ax.plot(x, np.nanmean(arr, 0), color=c, label={-2: "low to mixed", -1: "high to mixed"}[kind])
    ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("trials from block switch"); ax.set_ylabel(lab)
axes[0].legend(); plt.tight_layout(); plt.show()
for lab, dyn in (("wait", wd), ("initiation", idn)):
    arr = np.array([np.nanmean([dyn[r][k] for r in names], 0) for k in (-2, -1)])
    print(f"{lab}: plotted range {np.nanmin(arr):.3f} to {np.nanmax(arr):.3f}")

**Result: reproduced** (fig2a-wait-time-converges). Before the switch, wait times after low
blocks are 0.330 z above those after high blocks; 11 to 20 trials into the mixed block the
difference is 0.022, a ratio of 0.067, within the criterion's 0.25 (0.170 for trials 6 to 20).

**Result: not reproduced** (fig2b-initiation-previous-block), by the criterion as written.
Initiation times in mixed blocks depend strongly on the previous block, but with the sign
reversed: the difference (after low minus after high) is 0.121 z before the switch and -0.256
after it, a ratio of -2.12, and it is below zero in 290 of 291 rats (signed-rank p = 1.8e-49).
The criterion required the difference to stay above zero. The reversal (faster after low blocks,
slower after high blocks) is the incentive-contrast effect and overshoot that the paper describes
for Figure 2b, so the data agree with the paper's statement; the outcome comes from a criterion
written with the wrong sign before the analysis.

## fig2cd-regression-kernels

> Wait times are explained by the current reward, while trial initiation times reflect an
> exponentially weighted effect of previous rewards.

## fig2e-tau-differ

> The time constants of exponentials fit to the previous-reward coefficients differ between trial
> initiation and wait times (p = 1.30e-49, two-tailed signed-rank test, N = 291).

## fig2f-tau-uncorrelated

> The time constants of trial initiation and wait times are not correlated across rats.

In [ ]:
started = time.time()
reg_wt = [w.regress_wait_on_rewards(rats[r], 10) for r in names]
reg_i9 = [w.regress_initiation_on_rewards(rats[r], 9) for r in names]
reg_i10 = [w.regress_initiation_on_rewards(rats[r], 10) for r in names]
print(f"regressions and fits in {time.time() - started:.0f} s")
b_wt = np.array([b for b, _ in reg_wt]); tau_wt = np.array([t for _, t in reg_wt])
b_i9 = np.array([b for b, _ in reg_i9]); tau_i9 = np.array([t for _, t in reg_i9])
b_i10 = np.array([b for b, _ in reg_i10]); tau_i10 = np.array([t for _, t in reg_i10])
print("Mean wait time coefficients (current, 1 to 9 back; after zeroing):", np.round(b_wt[:, :10].mean(0), 4).tolist())
print("Mean initiation time coefficients (1 to 9 back; after zeroing):", np.round(b_i9[:, :9].mean(0), 4).tolist())
print(f"Wait time, current reward coefficient above zero: {np.sum(b_wt[:, 0] > 0)} rats, signed-rank p = {w.signrank(b_wt[:, 0]):.3g}")
print(f"Initiation time, previous reward coefficient below zero: {np.sum(b_i9[:, 0] < 0)} rats, signed-rank p = {w.signrank(b_i9[:, 0]):.3g}")
print(f"|mean wait coefficient 1 back| / mean current = {abs(b_wt[:, 1].mean()) / b_wt[:, 0].mean():.3f}")
mi = np.abs(b_i9[:, :9].mean(0))
print(f"|mean initiation coefficient| at 1, 3 and 6 back: {mi[0]:.4f}, {mi[2]:.4f}, {mi[5]:.4f}")
print("Rats with every previous wait time coefficient zeroed:", int(np.sum(np.all(b_wt[:, 1:10] == 0, 1))))

**Result: reproduced.** The current offer raises wait times (coefficient above zero in 289 of
291 rats, p = 3.8e-49), and the mean coefficient one trial back is 1.6% of it in magnitude;
initiation times fall with the previous reward (coefficient below zero in 249 rats, p = 1.3e-37),
with mean coefficients falling in magnitude from 0.0307 one trial back to 0.0107 three back and
0.0009 six back.

In [ ]:
print(f"Tau, wait times: median {np.median(tau_wt):.2f} (0 in {np.sum(tau_wt == 0)} rats); initiation times "
      f"(9 back, Methods): median {np.median(tau_i9):.2f} (0 in {np.sum(tau_i9 == 0)} rats)")
print(f"Signed-rank, initiation against wait time taus: p = {w.signrank(tau_i9, tau_wt):.3g}; "
      f"initiation larger in {np.sum(tau_i9 > tau_wt)} rats, smaller in {np.sum(tau_i9 < tau_wt)}")
print(f"Variant, 10 back (Figure2.m): median {np.median(tau_i10):.2f}, p = {w.signrank(tau_i10, tau_wt):.3g}")
print("Tau ranges: wait", tau_wt.min().round(2), "to", tau_wt.max().round(2), "; initiation", tau_i9.min().round(2), "to", tau_i9.max().round(2),
      "; initiation taus at the upper bound of 10:", int(np.sum(tau_i9 >= 9.99)))

**Result: reproduced.** Initiation time taus (median 1.63 trials) exceed wait time taus (median
0; in 248 rats no previous coefficient survives), larger in 262 rats and smaller in 5, p = 1.5e-44
against the paper's 1.30e-49. Nine initiation time taus are at the fit's upper bound of 10.

In [ ]:
r9, p9 = stats.pearsonr(tau_i9, tau_wt)
r10, p10 = stats.pearsonr(tau_i10, tau_wt)
print(f"Pearson, initiation (9 back, Methods) against wait time taus: r = {r9:.3f}, p = {p9:.3g}  (paper: r = -0.03, p = 0.53 "
      "in the legend; r = 0.08, p = 0.18 in the text)")
print(f"Variant, 10 back (Figure2.m): r = {r10:.3f}, p = {p10:.3g}")
nz = tau_wt > 0
r_nz, p_nz = stats.pearsonr(tau_i9[nz], tau_wt[nz])
print(f"Variant, the {nz.sum()} rats with a wait time tau above 0: r = {r_nz:.3f}, p = {p_nz:.3g}")

**Result: reproduced.** The two taus are not correlated (r = 0.023, p = 0.70); with the code's
10 previous rewards r = 0.023, p = 0.69. The paper gives two different values, r = -0.03 in the
legend and r = 0.08 in the text.

## fig2gh-fast-slow-groups

> Rats with fast and slow trial initiation time constants differ in their initiation time
> coefficients (p = 0.02, permutation test, N = 116) but not in their wait time coefficients
> (p = 0.1).

In [ ]:
rng = np.random.default_rng(2003)
for label, taus, bi in (("9 back (Methods)", tau_i9, b_i9), ("10 back (Figure2.m)", tau_i10, b_i10)):
    first = taus < np.percentile(taus, 20); last = taus > np.percentile(taus, 80)
    print(f"{label}: fast {first.sum()} rats, slow {last.sum()} rats, N = {first.sum() + last.sum()}")
    for name, betas in (("initiation", bi), ("wait", b_wt)):
        obs, p = w.tau_shuffle_test(first, last, betas, 500, rng)
        obs_c, p_c = w.tau_shuffle_test(first, last, betas, 500, rng, consistent=True)
        print(f"   {name} coefficients: tau difference {obs:.3f}, p = {p:.3f}; with the observed groups fit as the "
              f"shuffled ones, {obs_c:.3f}, p = {p_c:.3f}")

**Result: reproduced.** With 9 previous rewards the fast and slow groups have 57 and 58 rats (N
= 115; 116 with the code's 10). By the code's test the groups differ in their initiation time
coefficients (p = 0, no shuffle of 500 as extreme; paper p = 0.02) and not in their wait time
coefficients (p = 0.21; paper p = 0.1). The wait time result depends on the code's inconsistent
fits: when the observed groups are fit as the shuffled ones, the wait time p is also 0.

## fig2j-wait-time-previous-reward

> For most rats (253 of 291), wait times for 20 uL in mixed blocks did not differ with the previous
> reward offer; 38 rats had p < 0.05, and the population p = 0.8.

## fig2k-initiation-previous-reward

> For 89% of rats (256 of 291) trial initiation times in mixed blocks were modulated by the
> previous reward (p < 0.05 each); population p = 1.84e-49.

## fig2l-sensitivity-differs

> Sensitivity to the previous offer is larger for trial initiation times than for wait times
> (p = 6.21e-38, two-tailed signed-rank test, N = 291).

In [ ]:
pw_ = np.array([w.wait_by_previous_offer(rats[r]) for r in names])
print(f"Wait time, 20 uL catch trials in mixed blocks: rats with p < 0.05: {np.sum(pw_[:, 2] < 0.05)} (expected 38 +- 3.8)")
print(f"Population, rank-sum of per-rat means (legend): p = {w.ranksum(pw_[:, 0], pw_[:, 1]):.3g}; "
      f"signed-rank (Figure2.m, variant): p = {w.signrank(pw_[:, 0], pw_[:, 1]):.3g}")
print(f"Mean z after offers below 20: {np.nanmean(pw_[:, 0]):.3f}, above 20: {np.nanmean(pw_[:, 1]):.3f}")
print("Trials per rat in the test, below/above:", end=" ")
cnt = []
for r in names:
    A = rats[r]; idx = np.flatnonzero((A["reward"] == 20) & (A["block"] == 1) & A["optout"] & A["catch"]); idx = idx[idx > 0]
    cnt.append((np.sum(A["reward"][idx - 1] < 20), np.sum(A["reward"][idx - 1] > 20)))
cnt = np.array(cnt); print("median", np.median(cnt, 0).tolist(), "; fewest", cnt.min(0).tolist())

**Result: reproduced.** 38 rats have p < 0.05, the paper's count, and the population rank-sum
test finds no effect (p = 0.71; the code's signed-rank p = 0.81, the paper's 0.8).

In [ ]:
pi20 = np.array([w.initiation_by_previous_offer(rats[r], True) for r in names])
piall = np.array([w.initiation_by_previous_offer(rats[r], False) for r in names])
for label, arr in (("trials offering 20 uL (Methods)", pi20), ("all mixed-block trials (Figure2.m, variant)", piall)):
    print(f"{label}: rats with p < 0.05: {np.sum(arr[:, 2] < 0.05)} (expected 256 +- 25.6); slower after offers below 20 in "
          f"{np.sum(arr[:, 0] > arr[:, 1])} rats")
    print(f"   population rank-sum (legend): p = {w.ranksum(arr[:, 0], arr[:, 1]):.3g}; signed-rank: p = {w.signrank(arr[:, 0], arr[:, 1]):.3g}")
nc = np.array([w.initiation_by_previous_offer(no_cue[r], True) for r in names])
print(f"Variant without pre-cue sessions, 20 uL trials: {np.sum(nc[:, 2] < 0.05)} rats with p < 0.05")

**Result: reproduced.** On trials offering 20 uL, as the Methods state, 265 rats have p < 0.05,
within 256 +- 25.6, and the population rank-sum test finds slower initiation after offers below
20 uL (p = 9.7e-25). The code's reading, all mixed-block trials, gives 255 rats and the paper's
p = 1.84e-49 by signed rank.

In [ ]:
sens_wt = pw_[:, 0] - pw_[:, 1]
for label, arr in (("20 uL trials (Methods)", pi20), ("all mixed-block trials (code, variant)", piall)):
    sens_i = arr[:, 0] - arr[:, 1]
    print(f"{label}: sensitivity median, initiation {np.nanmedian(sens_i):.3f}, wait {np.nanmedian(sens_wt):.3f}; "
          f"signed-rank p = {w.signrank(sens_i, sens_wt):.3g}; initiation larger in {np.sum(sens_i > sens_wt)} rats")

**Result: reproduced.** The sensitivity to the previous offer is larger for initiation than for
wait times (p = 2.2e-08 on 20 uL trials, larger in 183 rats); with the code's all mixed-block
trials p = 6.09e-38, the paper's 6.21e-38.

## fig3c-catch-probability

> Wait times in mixed blocks depend in a graded way on the catch probability; the cohorts held 3,
> 183, 61, 151 and 39 rats at catch probabilities 0.1, 0.15, 0.2, 0.25 and 0.35.

In [ ]:
PROBS = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35]
by_p, n_with, n_any = {}, {}, {}
for p in PROBS:
    m = []
    for r in names:
        A = rats[r]; keep = np.arange(len(A["reward"])) >= 1000
        use = (A["block"] == 1) & (A["prob_catch"] == p) & A["catch"] & keep
        m.append([np.nanmean(A["wait_time"][use & (A["reward"] == v)]) if (use & (A["reward"] == v)).any() else np.nan
                  for v in w.STANDARD_VOLUMES])
    m = np.array(m)
    by_p[p] = m
    n_with[p] = int(np.sum(np.any(~np.isnan(m), 1)))
    n_any[p] = int(np.sum([np.any(rats[r]["prob_catch"] == p) for r in names]))
table = pd.DataFrame({"rats (code's selection)": n_with, "rats with any trial": n_any,
                      "paper": {0.1: 3, 0.15: 183, 0.2: 61, 0.25: 151, 0.3: None, 0.35: 39},
                      "mean wait (s), over rewards": {p: np.nanmean(np.nanmean(by_p[p], 0)) for p in PROBS}})
for i, v in enumerate(w.STANDARD_VOLUMES):
    table[f"{v} uL"] = [np.nanmean(by_p[p][:, i]) for p in PROBS]
print(table.round(2).to_string())
means = [np.nanmean(np.nanmean(by_p[p], 0)) for p in (0.15, 0.2, 0.25, 0.35)]
print("Strictly falling from 0.15 to 0.2 to 0.25 to 0.35:", bool(np.all(np.diff(means) < 0)))
# Within rats that had two probabilities: paired comparison of mean wait (all rewards) at 0.15 and 0.25.
pair = [(np.nanmean(by_p[0.15][i]), np.nanmean(by_p[0.25][i])) for i in range(len(names))
        if np.any(~np.isnan(by_p[0.15][i])) and np.any(~np.isnan(by_p[0.25][i]))]
pair = np.array(pair)
print(f"Rats with both 0.15 and 0.25: {len(pair)}; shorter at 0.25 in {np.sum(pair[:, 1] < pair[:, 0])}, "
      f"signed-rank p = {w.signrank(pair[:, 1], pair[:, 0]):.3g}")

**Result: reproduced.** By the code's selection the counts are 3, 188, 61, 151 and 40 rats at
catch probabilities 0.1 to 0.35, each within tolerance of the paper's 3, 183, 61, 151 and 39, and
the mean wait time falls strictly from 12.93 s at 0.15 to 11.77, 11.31 and 10.60 s at 0.2, 0.25
and 0.35. Within the 59 rats with both 0.15 and 0.25, 56 wait less at 0.25 (p = 7.7e-10). The files
also hold a probability of 0.3 (8 rats by the code's selection), which the paper does not list.

## fig3j-model-comparison

> Wait times are better fit by the inferential model, which uses hidden state inference, than by
> the retrospective model (BIC, p = 1.07e-37, two-tailed signed-rank test, N = 291).

## figs9-misinference

> Wait times for 20 uL are shorter when the inferential model infers a high block than when it
> infers a low block (p << 0.001, signed-rank test, N = 291).

## fig3l-inference-quality

> Rats with good and poor inference (lambda above the 80th or below the 20th percentile) differ in
> wait time dynamics at transitions from mixed into low or high blocks, but not in trial
> initiation time dynamics.

The models are fit per rat in four processes; each rat's random starts come from its own
generator, seeded from the claim's seed and the rat's index.

In [ ]:
ALPHAS = np.geomspace(1e-3, 1, 40)
LAMBDAS = np.linspace(0, 1, 11)

def fit_rat(item):
    M, seed = item
    rng = np.random.default_rng(seed)
    b = w.beliefs(M)
    inferred = b.argmax(1)
    p_inf, nll_inf = w.fit_inferential(M, inferred, rng, starts=20)
    p_ret, nll_ret = w.fit_retrospective(M, b[:, 0], ALPHAS)
    lam_b = w.beliefs(M, LAMBDAS)
    lam_nll = [w.fit_inferential(M, lam_b[k].argmax(1), rng, starts=0, x0=p_inf)[1] for k in range(len(LAMBDAS))]
    return {"p_inf": p_inf, "nll_inf": nll_inf, "p_ret": p_ret, "nll_ret": nll_ret, "lam_nll": np.array(lam_nll),
            "inferred": inferred, "n_fit": int(M["fitted"].sum()), "n_trials": M["n_trials"]}

started = time.time()
inputs = {r: w.model_inputs(rats[r]) for r in names}
seeds = np.random.SeedSequence(2004).spawn(len(names))
with ProcessPoolExecutor(4) as pool:
    fits = dict(zip(names, pool.map(fit_rat, [(inputs[r], s) for r, s in zip(names, seeds)], chunksize=4)))
print(f"fit {len(fits)} rats in {time.time() - started:.0f} s")

In [ ]:
n_fit = np.array([fits[r]["n_fit"] for r in names]); n_all = np.array([fits[r]["n_trials"] for r in names])
nll_inf = np.array([fits[r]["nll_inf"] for r in names]); nll_ret = np.array([fits[r]["nll_ret"] for r in names])
for label, n in (("n = fitted trials", n_fit), ("variant, n = all trials", n_all)):
    bic_inf = 4 * np.log(n) + 2 * nll_inf; bic_ret = 2 * np.log(n) + 2 * nll_ret
    d = bic_ret - bic_inf
    print(f"{label}: delta BIC (retrospective - inferential) median {np.median(d):.1f}, range {d.min():.1f} to {d.max():.1f}; "
          f"inferential better in {np.sum(d > 0)} of {len(d)} rats; signed-rank p = {w.signrank(bic_ret, bic_inf):.3g}")
aic_d = (2 * 2 + 2 * nll_ret) - (2 * 4 + 2 * nll_inf)
print(f"AIC: inferential better in {np.sum(aic_d > 0)} rats, p = {w.signrank(aic_d):.3g}")
print("Fitted trials per rat:", int(n_fit.min()), "to", int(n_fit.max()))
p_ret = np.array([fits[r]["p_ret"] for r in names])
print("Retrospective baseline learning rate, median", np.median(p_ret[:, 0]).round(3), "; at the grid's ends:",
      int(np.sum(p_ret[:, 0] <= ALPHAS[0])), "low,", int(np.sum(p_ret[:, 0] >= ALPHAS[-1])), "high")
p_inf = np.array([fits[r]["p_inf"] for r in names])
print("Inferential parameters at a bound (1e-4 or 1):", int(np.sum(np.any((p_inf <= 1.0001e-4) | (p_inf >= 0.9999), 1))), "rats")

**Result: reproduced.** The inferential model has the lower BIC in 291 of 291 rats (median
difference 847), p = 1.8e-49 by signed rank (paper 1.07e-37); counting all trials in n, or using
AIC, gives the same.

In [ ]:
mis_hi, mis_lo = [], []
for r in names:
    A, f = rats[r], fits[r]
    good = ~A["vios"]
    inf = np.full(len(A["reward"]), -1); inf[good] = f["inferred"]
    sel = (A["block"] == 1) & A["catch"] & (A["reward"] == 20)
    mis_hi.append(np.nanmean(A["wait_time"][sel & (inf == 1)]) if (sel & (inf == 1)).any() else np.nan)
    mis_lo.append(np.nanmean(A["wait_time"][sel & (inf == 2)]) if (sel & (inf == 2)).any() else np.nan)
mis_hi, mis_lo = np.array(mis_hi), np.array(mis_lo)
ok = ~np.isnan(mis_hi - mis_lo)
print(f"Rats with both inferences on 20 uL mixed-block catch trials: {ok.sum()}")
print(f"Mean wait, inferred high {np.nanmean(mis_hi):.2f} s, inferred low {np.nanmean(mis_lo):.2f} s; shorter when inferred high in "
      f"{np.sum(mis_hi[ok] < mis_lo[ok])} rats; signed-rank p = {w.signrank(mis_hi, mis_lo):.3g}")

**Result: reproduced.** In mixed blocks, wait times for 20 uL are shorter when the model infers
a high block (mean 10.87 s) than a low block (11.91 s), in 230 of 291 rats, p = 1.1e-22.

In [ ]:
lam_nll = np.array([fits[r]["lam_nll"] for r in names])
lam = LAMBDAS[lam_nll.argmin(1)]
print("Fitted lambda, rats per grid value:", dict(zip(LAMBDAS.round(1).tolist(), np.bincount(lam_nll.argmin(1), minlength=len(LAMBDAS)).tolist())))
wd30 = {r: w.wait_dynamics(rats[r], 30, 10) for r in names}
id30 = {r: w.initiation_dynamics(rats[r], 30, 10) for r in names}

def shuffle_job(job):
    lo_rows, hi_rows, n_perm, seed = job
    return w.lambda_shuffle_test(lo_rows, hi_rows, n_perm, 30, np.random.default_rng(seed))

jobs, labels = [], []
splits = (("20th/80th (Methods), ties at the cut included; 500 shuffles", 20, 80),
          ("40th/60th (Figure3.m, variant); 200 shuffles", 40, 60))
for k, (label, lo_pct, hi_pct) in enumerate(splits):
    lo_g = lam <= np.percentile(lam, lo_pct); hi_g = lam >= np.percentile(lam, hi_pct)
    if lo_pct == 40:
        lo_g = lam < np.percentile(lam, lo_pct); hi_g = lam > np.percentile(lam, hi_pct)
    print(f"{label}: poor inference {lo_g.sum()} rats (lambda <= {lam[lo_g].max():.1f}), good {hi_g.sum()} rats (lambda >= {lam[hi_g].min():.1f})")
    for meas, dyn in (("wait", wd30), ("initiation", id30)):
        for kind, tr in ((2, "mixed to low"), (1, "mixed to high")):
            arr = np.array([dyn[r][kind] for r in names])
            jobs.append((arr[lo_g], arr[hi_g], 500 if k == 0 else 200, 2005 + len(jobs)))
            labels.append({"split": label.split(" ")[0], "measure": meas, "transition": tr})
started = time.time()
with ProcessPoolExecutor(4) as pool:
    out = list(pool.map(shuffle_job, jobs))
print(f"shuffles in {time.time() - started:.0f} s")
res = pd.DataFrame([{**l, "slope diff": d[1], "p slope": p[1], "right asymptote diff": d[0], "p right asymptote": p[0]}
                    for l, (d, p) in zip(labels, out)])
print(res.round(4).to_string(index=False))

**Result: reproduced.** Lambda fit on the grid lies at 0 for 61 rats and at 1 for 108, so the
20th and 80th percentiles fall on these ends and the groups are those rats (61 and 108). For wait
times the right asymptote from mixed to high blocks differs between them (p = 0.003, the paper's
value), while the slope from mixed to low does not (p = 0.34; paper 0.02). For initiation times
neither differs (slope from mixed to low p = 0.32, right asymptote from mixed to high p = 0.11).
With the code's 40th and 60th percentiles (110 and 108 rats), the initiation time slope from mixed
to high differs (p = 0.0075), which that split would count against the claim.

## fig3h-dynamic-learning-rate (blocked)

> A retrospective model of trial initiation times needs a learning rate scaled by changes in the
> inferred block to capture both the slow and fast dynamics at block transitions; a static or
> RPE-scaled learning rate does not.

Blocked, methods underspecified. The paper did not fit the trial initiation time models; it
simulated them with hand-chosen parameters on one rat's trial sequence (Figure3.m, FigureS7.m)
and judged by eye which captures the dynamics. Neither the paper nor the code states a measure of
"capturing" the dynamics that a test could apply.

## fig4de-overshoot-learning (blocked)

Blocked, data missing. Figure 4d and 4e use the per-session raw structs (`S_Structs`), which hold
the sessions of training the processed structs leave out. They are not in the dandiset, which holds
only `A_Structs_Final`, nor in the Zenodo deposit, whose zip holds `A_Structs_Final`,
`A_Structs_all`, `ModelFits` and `ratList.mat` only.

## fig5b-precue-offer

> With the reward cue played before trial initiation, trial initiation times became sensitive to
> the offered reward.

## fig5c-precue-previous-reward

> In the pre-initiation cue task, trial initiation times for 20 uL in mixed blocks were still
> modulated by the previous reward (p < 0.05 for 13 of 16 rats; population p = 1.03e-4).

## fig5d-precue-previous-block

> In the pre-initiation cue task, trial initiation times in mixed blocks still depended on the
> previous block.

In [ ]:
cue_rats = sorted(r for r in names if rats[r]["cue"].any())
cue_data = {r: w.subset(rats[r], rats[r]["cue"]) for r in cue_rats}
orig_data = {r: w.subset(rats[r], ~rats[r]["cue"]) for r in cue_rats if (~rats[r]["cue"]).any()}

def offer_slope(A, drop_post_violation):
    L = w.zscore(w.clean_iti(A["iti"]))
    use = (A["block"] == 1) & ~np.isnan(L)
    if drop_post_violation:
        use &= ~np.r_[False, A["vios"][:-1]]
    return np.polyfit(np.log2(A["reward"][use]), L[use], 1)[0]

slope_cue = pd.Series({r: offer_slope(cue_data[r], False) for r in cue_rats})
slope_orig = pd.Series({r: offer_slope(orig_data[r], True) for r in orig_data})
print(pd.DataFrame({"cue sessions": pd.Series({r: cue_data[r]["session"].max() + 1 for r in cue_rats}),
                    "slope, cue": slope_cue, "slope, original": slope_orig}).round(3).to_string())
print(f"Cue slopes below zero: {np.sum(slope_cue < 0)} of {len(slope_cue)}, signed-rank p = {w.signrank(slope_cue):.3g}")
both_ = slope_orig.index
print(f"Cue below original, {len(both_)} rats: {np.sum(slope_cue[both_] < slope_orig)}; signed-rank p = "
      f"{w.signrank(slope_cue[both_], slope_orig):.3g}")

**Result: reproduced.** In cue sessions initiation times fall with the offered volume in all 16
rats (signed-rank p = 3.1e-05), and in all 12 rats with sessions of the original task the slope is
steeper in cue sessions than in the original task (p = 0.00049).

In [ ]:
def precue_previous(data, only_20ul):
    arr = pd.DataFrame({r: w.initiation_by_previous_offer(data[r], only_20ul) for r in data}, index=["below", "above", "p"]).T
    return arr

late_data = {}
for r in cue_rats:
    A = rats[r]
    if (~A["cue"]).any():
        cue_sess = np.unique(A["session"][A["cue"]])
        keep = A["cue"] & np.isin(A["session"], cue_sess[30:])
    else:
        keep = A["cue"]
    if keep.any():
        late_data[r] = w.subset(A, keep)
for label, data, only in (("all cue sessions, 20 uL trials (main)", cue_data, True),
                          ("all cue sessions, all mixed trials (code, variant)", cue_data, False),
                          ("after each switched rat's first 30 cue sessions, 20 uL trials (variant)", late_data, True)):
    arr = precue_previous(data, only)
    print(f"{label}: {len(arr)} rats; p < 0.05 in {np.sum(arr.p < 0.05)} (expected 13 +- 1.3); slower after offers below 20 in "
          f"{np.sum(arr.below > arr.above)}; population rank-sum p = {w.ranksum(arr.below, arr.above):.3g}, signed-rank p = "
          f"{w.signrank(arr.below, arr.above):.3g}")
print(precue_previous(cue_data, True).round(4).to_string())

**Result: reproduced.** In all cue sessions, 14 of 16 rats have p < 0.05 on 20 uL trials, within
13 +- 1.3, and the population rank-sum test finds slower initiation after offers below 20 uL (p =
0.04; 12 of 16 rats in that direction). The result is fragile: the code's reading (all
mixed-block trials) gives 12 rats (p = 1.9e-06), and keeping only sessions after each switched
rat's first 30 cue sessions leaves 11 rats, 10 with p < 0.05, and a population p of 0.47.

In [ ]:
cue_dyn = {r: w.initiation_dynamics(cue_data[r]) for r in cue_rats}
_ = transition_summary(cue_dyn, "Initiation time, cue sessions (fig5d)")
orig_dyn = {r: w.initiation_dynamics(orig_data[r]) for r in orig_data}
_ = transition_summary(orig_dyn, "Same rats, original task (reference)")

**Result: not reproduced**, by the criterion as written. As in fig2b, initiation times in the
mixed block depend on the previous block with the sign reversed: the difference is 0.105 z before
the switch and -0.263 after (ratio -2.51), below zero in 15 of 16 rats (p = 9.2e-05). The same
rats in the original task show the same reversal (ratio -2.83). The criterion required the sign to
persist; the data agree with the paper's statement of a dependence on the previous block.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| dataset-counts | reproduced |
| zenodo-link | partially reproduced |
| fig1f-wait-time-block | reproduced |
| fig1i-initiation-block | reproduced |
| figs2-unrewarded-drive | reproduced |
| figs12-no-sex-difference | reproduced |
| fig2a-wait-time-converges | reproduced |
| fig2b-initiation-previous-block | not reproduced (criterion's sign) |
| fig2cd-regression-kernels | reproduced |
| fig2e-tau-differ | reproduced |
| fig2f-tau-uncorrelated | reproduced |
| fig2gh-fast-slow-groups | reproduced |
| fig2j-wait-time-previous-reward | reproduced |
| fig2k-initiation-previous-reward | reproduced |
| fig2l-sensitivity-differs | reproduced |
| fig3c-catch-probability | reproduced |
| fig3j-model-comparison | reproduced |
| figs9-misinference | reproduced |
| fig3l-inference-quality | reproduced |
| fig3h-dynamic-learning-rate | blocked (methods underspecified) |
| fig4a-wait-ratio-learning | partially reproduced (second notebook) |
| fig4bc-regression-learning | reproduced (second notebook) |
| fig4de-overshoot-learning | blocked (data missing) |
| fig5b-precue-offer | reproduced |
| fig5c-precue-previous-reward | reproduced |
| fig5d-precue-previous-block | not reproduced (criterion's sign) |

**How the analysis differs from the paper's.** The per-rat measures are recomputed from the
dandiset with the paper's MATLAB code as the guide. Where the Methods and the code disagree, the
Methods give the main result and the code's choice is shown: z-scored against raw means for
Figure 1, 9 against 10 previous rewards, trials offering 20 uL against all mixed-block trials,
rank-sum against signed-rank tests in Figure 2j and 2k, and the percentile split of Figure 3l.
The models of Figure 3 are refit here: from 20 random starts (the paper used 100), on catch-trial
wait times with beliefs updated over non-violation trials (the published code does not show the
fitting), without cross-validation, and with lambda on a grid of 11 values, which ties many rats
at 0 and 1. Exponentials are fit by a grid over tau in place of fmincon, and logistic fits use
scipy. The pre-cue claims use the cue sessions the dandiset holds, which passed the paper's
inclusion criteria, where the paper used all of them. Two criteria (fig2b and fig5d) were written
with the wrong sign for the dependence on the previous block; the data show the dependence, with
the incentive-contrast reversal the paper describes.

**What was found about the dandiset.** It holds the paper's processed data for all 291 rats, with
the same values as the Zenodo structs, plus the raw Bpod events, which agree with the processed
times. Nine sessions of the structs (0.035% of trials) are missing; one session (L011) is deposited
twice; sessions of 10 rats are in a different order from the structs. The processed table lacks
the catch probability (a typo in the conversion's column mapping); the training stage is coded 9
without saying that it is the paper's stage 8; ages are missing for 275 rats; sounds are stored
without their identity; 456 sessions start with a high or low block; and the 762 pre-cue sessions
of Figure 5 are among the main data, marked only by a task argument. The early training sessions
that Figure 4 uses are not in the dandiset (they are on Zenodo), and the raw structs behind Figure
4d, 4e and 5 are in neither.